# Notebook 09k: ViT-Small TransReID CityFlowV2 Training
**Multi-Camera Tracking System -- Kaggle Training Pipeline**

Train a secondary **TransReID ViT-Small/16** model for CityFlowV2 vehicle re-identification.

## Recipe
- Backbone: `vit_small_patch16_224.augreg_in21k_ft_in1k`
- TransReID tricks: **SIE**, **JPM**, **overlap patch embedding**
- Input: **224x224** with **ImageNet normalization**
- Loss: **CE+LS(0.1) + Triplet(m=0.3) + Center(5e-4, start epoch 15)**
- Optimizer: **AdamW**, backbone LR = `0.1x`, cosine annealing after 10-epoch warmup
- Batch recipe: **P=16, K=4** (`64` images per batch)

## Output
- `/kaggle/working/vit_small_transreid_cityflowv2_best.pth`
- `/kaggle/working/vit_small_transreid_cityflowv2_metadata.json`

In [ ]:
!pip install -q timm matplotlib pandas loguru

In [ ]:
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, Sampler

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
NUM_GPUS = max(torch.cuda.device_count(), 1)
print(f"GPUs available: {NUM_GPUS}")
for gpu_index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(gpu_index)
    print(f"  GPU {gpu_index}: {torch.cuda.get_device_name(gpu_index)} ({props.total_memory / 1024**3:.1f} GB)")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("/kaggle/working")
CURVE_PATH = OUTPUT_DIR / "vit_small_transreid_cityflowv2_curves.png"
CHECKPOINT_PATH = OUTPUT_DIR / "vit_small_transreid_cityflowv2_best.pth"
LAST_CHECKPOINT_PATH = OUTPUT_DIR / "vit_small_transreid_cityflowv2_last.pth"
METADATA_EXPORT_PATH = OUTPUT_DIR / "vit_small_transreid_cityflowv2_metadata.json"
CROP_DIR = Path("/tmp/cityflowv2_crops_09k")
CROP_DIR.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "vit_model": "vit_small_patch16_224.augreg_in21k_ft_in1k",
    "img_size": 224,
    "resize_size": 240,
    "embed_dim": 384,
    "overlap_patch_stride": 12,
    "num_workers": 4,
    "p": 16,
    "k": 4,
    "batch_size": 64,
    "epochs": 120,
    "warmup_epochs": 10,
    "eval_every": 10,
    "label_smoothing": 0.1,
    "triplet_margin": 0.3,
    "center_weight": 5e-4,
    "center_start_epoch": 15,
    "jpm_loss_weight": 0.5,
    "backbone_lr": 3.5e-5,
    "head_lr": 3.5e-4,
    "weight_decay": 5e-4,
    "llrd_decay": 0.75,
    "grad_clip_norm": 5.0,
    "max_crops_per_id_cam": 20,
    "min_area": 2000,
    "min_bbox_side": 30,
    "train_ratio": 0.7,
    "min_cams_for_eval": 2,
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std": [0.229, 0.224, 0.225],
}

print(json.dumps(CONFIG, indent=2))

## 1. Locate CityFlowV2 Dataset

This notebook uses the same Kaggle-mounted CityFlowV2 dataset source as the other vehicle training notebooks.

In [ ]:
candidate_mounts = [
    Path("/kaggle/input/data-aicity-2023-track-2"),
    Path("/kaggle/input/datasets/thanhnguyenle/data-aicity-2023-track-2"),
]

CITYFLOW_DIR = None
for candidate in candidate_mounts:
    if candidate.exists():
        CITYFLOW_DIR = candidate
        break

if CITYFLOW_DIR is None:
    raise FileNotFoundError(
        "CityFlowV2 dataset not found at any candidate mount:\n"
        + "\n".join(f"  - {path}" for path in candidate_mounts)
    )

RAW_ROOT = None
for subpath in ["AIC22_Track1_MTMC_Tracking/train", "train"]:
    probe = CITYFLOW_DIR / subpath
    if probe.exists():
        RAW_ROOT = probe
        break

if RAW_ROOT is None:
    raise FileNotFoundError(f"CityFlowV2 raw train split not found under {CITYFLOW_DIR}")

camera_dirs = sorted(path for path in RAW_ROOT.glob("S*/c*") if path.is_dir())
CAMERAS = [f"{path.parent.name}_{path.name}" for path in camera_dirs]
cam_dir_map = {f"{path.parent.name}_{path.name}": path for path in camera_dirs}

print(f"CityFlowV2 dataset root: {CITYFLOW_DIR}")
print(f"Raw train root: {RAW_ROOT}")
print(f"Cameras with GT: {len(CAMERAS)}")
print(CAMERAS[:8], "..." if len(CAMERAS) > 8 else "")

## 2. Build CityFlowV2 ReID Splits

Extract crops from GT/video pairs, then build train/query/gallery with the same CityFlowV2 cross-camera protocol used by the existing ViT notebooks.

In [ ]:
def load_gt(gt_path):
    rows = []
    with open(gt_path, encoding="utf-8") as handle:
        for line in handle:
            parts = line.strip().split(",")
            if len(parts) < 6:
                continue
            frame_id = int(parts[0])
            track_id = int(parts[1])
            x = int(float(parts[2]))
            y = int(float(parts[3]))
            w = int(float(parts[4]))
            h = int(float(parts[5]))
            rows.append((frame_id, track_id, x, y, w, h))
    return rows


def extract_crops_from_camera(cam_name, cam_dir, crop_dir, max_crops, min_area, min_bbox_side):
    gt_file = cam_dir / "gt" / "gt.txt"
    video_file = cam_dir / "vdo.avi"
    if not gt_file.exists() or not video_file.exists():
        print(f"  SKIP {cam_name}: missing GT or video")
        return {}

    gt_rows = load_gt(gt_file)
    if not gt_rows:
        print(f"  SKIP {cam_name}: empty GT")
        return {}

    id_to_detections = defaultdict(list)
    for frame_id, track_id, x, y, w, h in gt_rows:
        if w * h < min_area or w < min_bbox_side or h < min_bbox_side:
            continue
        id_to_detections[track_id].append((frame_id, x, y, w, h))

    sampled = {}
    for track_id, detections in id_to_detections.items():
        detections.sort(key=lambda item: item[0])
        if len(detections) <= max_crops:
            sampled[track_id] = detections
        else:
            sample_idx = np.linspace(0, len(detections) - 1, num=max_crops, dtype=int)
            sampled[track_id] = [detections[index] for index in sample_idx]

    detections_by_frame = defaultdict(list)
    for track_id, detections in sampled.items():
        for frame_id, x, y, w, h in detections:
            detections_by_frame[frame_id].append((track_id, x, y, w, h))

    crops = defaultdict(list)
    cap = cv2.VideoCapture(str(video_file))
    if not cap.isOpened():
        print(f"  SKIP {cam_name}: failed to open video")
        return {}

    saved = 0
    for frame_id in sorted(detections_by_frame):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id - 1)
        ok, frame = cap.read()
        if not ok:
            continue
        frame_h, frame_w = frame.shape[:2]
        for track_id, x, y, w, h in detections_by_frame[frame_id]:
            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(frame_w, x + w)
            y2 = min(frame_h, y + h)
            if x2 <= x1 or y2 <= y1:
                continue
            crop = frame[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            file_name = f"{track_id:04d}_{cam_name}_f{frame_id:06d}.jpg"
            file_path = crop_dir / file_name
            if cv2.imwrite(str(file_path), crop):
                crops[track_id].append(str(file_path))
                saved += 1

    cap.release()
    print(f"  {cam_name}: {saved} crops from {len(crops)} vehicles")
    return dict(crops)


existing_crops = list(CROP_DIR.glob("*.jpg"))
if len(existing_crops) > 100:
    print(f"Reusing {len(existing_crops)} crops from {CROP_DIR}")
    all_crops = defaultdict(lambda: defaultdict(list))
    for file_path in sorted(existing_crops):
        parts = file_path.stem.split("_")
        track_id = int(parts[0])
        cam_name = parts[1] + "_" + parts[2]
        all_crops[track_id][cam_name].append(str(file_path))
    all_crops = {track_id: dict(cam_map) for track_id, cam_map in all_crops.items()}
else:
    for file_path in CROP_DIR.glob("*.jpg"):
        file_path.unlink()
    all_crops = defaultdict(lambda: defaultdict(list))
    for cam_name in CAMERAS:
        cam_crops = extract_crops_from_camera(
            cam_name,
            cam_dir_map[cam_name],
            CROP_DIR,
            CONFIG["max_crops_per_id_cam"],
            CONFIG["min_area"],
            CONFIG["min_bbox_side"],
        )
        for track_id, paths in cam_crops.items():
            all_crops[track_id][cam_name].extend(paths)
    all_crops = {track_id: dict(cam_map) for track_id, cam_map in all_crops.items()}

total_crops = sum(sum(len(paths) for paths in cam_map.values()) for cam_map in all_crops.values())
multi_cam_ids = sorted(track_id for track_id, cam_map in all_crops.items() if len(cam_map) >= CONFIG["min_cams_for_eval"])
single_cam_ids = sorted(track_id for track_id, cam_map in all_crops.items() if len(cam_map) < CONFIG["min_cams_for_eval"])

rng = np.random.RandomState(SEED)
rng.shuffle(multi_cam_ids)
split_index = int(len(multi_cam_ids) * CONFIG["train_ratio"])
train_ids = set(multi_cam_ids[:split_index])
eval_ids = set(multi_cam_ids[split_index:])

cam_names = sorted({cam_name for cam_map in all_crops.values() for cam_name in cam_map})
cam2id = {cam_name: idx for idx, cam_name in enumerate(cam_names)}

train_pid_set = sorted(train_ids)
pid2label = {track_id: idx for idx, track_id in enumerate(train_pid_set)}
eval_pid2label = {track_id: idx for idx, track_id in enumerate(sorted(eval_ids))}

train_data = []
query_data = []
gallery_data = []

for track_id in sorted(train_ids):
    label = pid2label[track_id]
    for cam_name, paths in all_crops[track_id].items():
        camid = cam2id[cam_name]
        for path in paths:
            train_data.append((path, label, camid))

for track_id in sorted(eval_ids):
    pid = eval_pid2label[track_id]
    for cam_name, paths in all_crops[track_id].items():
        if not paths:
            continue
        camid = cam2id[cam_name]
        query_index = rng.randint(0, len(paths))
        query_data.append((paths[query_index], pid, camid))
        for path_index, path in enumerate(paths):
            if path_index != query_index:
                gallery_data.append((path, pid, camid))

distractor_pid = len(eval_ids)
for track_id in sorted(single_cam_ids):
    for cam_name, paths in all_crops[track_id].items():
        camid = cam2id[cam_name]
        for path in paths:
            gallery_data.append((path, distractor_pid, camid))
    distractor_pid += 1

num_classes = max(len(train_pid_set), 1)
num_cameras = max(len(cam_names), 1)
can_eval = bool(query_data) and bool(gallery_data)

print(f"Total crops: {total_crops}")
print(f"Train:   {len(train_data)} images, {num_classes} IDs")
print(f"Query:   {len(query_data)} images, {len(eval_ids)} IDs")
print(f"Gallery: {len(gallery_data)} images")
print(f"Cameras: {num_cameras}")
print(f"Evaluation possible: {can_eval}")
if not can_eval:
    raise RuntimeError("CityFlowV2 split generation failed to produce query/gallery data")

In [ ]:
train_tf = T.Compose([
    T.Resize((CONFIG["resize_size"], CONFIG["resize_size"]), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.Pad(10),
    T.RandomCrop((CONFIG["img_size"], CONFIG["img_size"])),
    T.ToTensor(),
    T.Normalize(mean=CONFIG["imagenet_mean"], std=CONFIG["imagenet_std"]),
    T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value="random"),
])

test_tf = T.Compose([
    T.Resize((CONFIG["img_size"], CONFIG["img_size"]), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=CONFIG["imagenet_mean"], std=CONFIG["imagenet_std"]),
])


class ReIDDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        path, pid, cam = self.data[index]
        image = Image.open(path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, pid, cam, path


class PKSampler(Sampler):
    def __init__(self, data_source, p=16, k=4):
        self.data_source = data_source
        self.p = p
        self.k = k
        self.pid_to_indices = defaultdict(list)
        for index, (_, pid, _) in enumerate(data_source):
            self.pid_to_indices[pid].append(index)
        self.pids = list(self.pid_to_indices.keys())
        self.batch_size = p * k

    def __iter__(self):
        shuffled_pids = self.pids[:]
        np.random.shuffle(shuffled_pids)
        batch = []
        for pid in shuffled_pids:
            indices = self.pid_to_indices[pid]
            selected = np.random.choice(indices, self.k, replace=len(indices) < self.k).tolist()
            batch.extend(selected)
            if len(batch) >= self.batch_size:
                yield from batch[: self.batch_size]
                batch = batch[self.batch_size :]

    def __len__(self):
        return len(self.pids) * self.k


train_ds = ReIDDataset(train_data, train_tf)
query_ds = ReIDDataset(query_data, test_tf)
gallery_ds = ReIDDataset(gallery_data, test_tf)

sampler = PKSampler(train_data, p=CONFIG["p"], k=CONFIG["k"])
train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG["batch_size"],
    sampler=sampler,
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
    drop_last=True,
)
query_loader = DataLoader(query_ds, batch_size=128, shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)
gallery_loader = DataLoader(gallery_ds, batch_size=128, shuffle=False, num_workers=CONFIG["num_workers"], pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Query batches: {len(query_loader)} | Gallery batches: {len(gallery_loader)}")

In [ ]:
class CrossEntropyLabelSmooth(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon

    def forward(self, inputs, targets):
        log_probs = F.log_softmax(inputs.float(), dim=1)
        with torch.no_grad():
            one_hot = torch.zeros_like(log_probs).scatter_(1, targets.unsqueeze(1), 1)
            smooth = (1 - self.epsilon) * one_hot + self.epsilon / self.num_classes
        return (-smooth * log_probs).sum(dim=1).mean()


class TripletLossHardMining(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)

    def forward(self, feats, pids):
        feats = F.normalize(feats.float(), p=2, dim=1)
        distances = torch.cdist(feats, feats, p=2)
        same_id = pids.unsqueeze(0).eq(pids.unsqueeze(1))
        different_id = ~same_id

        pos_dist = distances.clone()
        pos_dist[~same_id] = 0
        hardest_pos, _ = pos_dist.max(dim=1)

        neg_dist = distances.clone()
        neg_dist[~different_id] = float("inf")
        hardest_neg, _ = neg_dist.min(dim=1)

        target = torch.ones(feats.size(0), device=feats.device)
        return self.ranking_loss(hardest_neg, hardest_pos, target)


class CenterLoss(nn.Module):
    def __init__(self, num_classes, feat_dim):
        super().__init__()
        self.centers = nn.Parameter(torch.randn(num_classes, feat_dim))

    def forward(self, feats, labels):
        feats = feats.float()
        centers_batch = self.centers[labels]
        return ((feats - centers_batch) ** 2).sum(dim=1).mean()


@torch.no_grad()
def extract_features(model, loader, device="cuda", flip=True):
    model.eval()
    all_feats = []
    all_pids = []
    all_cams = []
    for imgs, pids, cams, _ in loader:
        imgs = imgs.to(device)
        cam_ids = cams.to(device).long()
        feats = model(imgs, cam_ids=cam_ids)
        if flip:
            flip_feats = model(torch.flip(imgs, dims=[3]), cam_ids=cam_ids)
            feats = (feats + flip_feats) / 2.0
        feats = F.normalize(feats, p=2, dim=1)
        all_feats.append(feats.cpu().numpy())
        all_pids.append(pids.numpy())
        all_cams.append(cams.numpy())

    if not all_feats:
        feat_dim = CONFIG["embed_dim"]
        return np.zeros((0, feat_dim), dtype=np.float32), np.zeros(0, dtype=int), np.zeros(0, dtype=int)

    return (
        np.concatenate(all_feats, axis=0),
        np.concatenate(all_pids, axis=0),
        np.concatenate(all_cams, axis=0),
    )


def eval_market1501(distmat, q_pids, g_pids, q_cams, g_cams, max_rank=10):
    if distmat.shape[0] == 0 or distmat.shape[1] == 0:
        return 0.0, np.zeros(max_rank, dtype=np.float32)

    indices = np.argsort(distmat, axis=1)
    matches = (g_pids[indices] == q_pids[:, None]).astype(np.int32)
    all_cmc = []
    all_ap = []

    for query_index in range(distmat.shape[0]):
        order = indices[query_index]
        remove = (g_pids[order] == q_pids[query_index]) & (g_cams[order] == q_cams[query_index])
        raw_cmc = matches[query_index][~remove]
        if raw_cmc.sum() == 0:
            continue

        cmc = raw_cmc.cumsum()
        cmc[cmc > 1] = 1
        padded = np.zeros(max_rank, dtype=np.float32)
        upto = min(max_rank, len(cmc))
        padded[:upto] = cmc[:upto]
        if upto < max_rank and upto > 0:
            padded[upto:] = padded[upto - 1]
        all_cmc.append(padded)

        precision = raw_cmc.cumsum() / (np.arange(len(raw_cmc)) + 1.0)
        all_ap.append(float((precision * raw_cmc).sum() / raw_cmc.sum()))

    if not all_ap:
        return 0.0, np.zeros(max_rank, dtype=np.float32)

    return float(np.mean(all_ap)), np.mean(np.stack(all_cmc, axis=0), axis=0)


@torch.no_grad()
def evaluate_cityflow(model, query_loader, gallery_loader, device="cuda"):
    qf, qp, qc = extract_features(model, query_loader, device=device, flip=True)
    gf, gp, gc = extract_features(model, gallery_loader, device=device, flip=True)
    distmat = 1.0 - qf @ gf.T
    mAP, cmc = eval_market1501(distmat, qp, gp, qc, gc, max_rank=10)
    return {
        "mAP": float(mAP),
        "rank1": float(cmc[0]) if len(cmc) > 0 else 0.0,
        "rank5": float(cmc[4]) if len(cmc) > 4 else float(cmc[-1]) if len(cmc) else 0.0,
        "rank10": float(cmc[9]) if len(cmc) > 9 else float(cmc[-1]) if len(cmc) else 0.0,
    }


print("Losses and CityFlowV2 evaluation ready")

## 3. TransReID ViT-Small/16

The model below keeps the same SIE/JPM flow as the existing CityFlowV2 ViT notebook, but swaps in a ViT-Small backbone and replaces the default patch embedding with an overlap patch embedding (`kernel=16`, `stride=12`).

In [ ]:
class OverlapPatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, stride=12, in_chans=3, embed_dim=384, bias=True, norm_layer=None):
        super().__init__()
        self.img_size = (img_size, img_size) if isinstance(img_size, int) else tuple(img_size)
        self.patch_size = (patch_size, patch_size) if isinstance(patch_size, int) else tuple(patch_size)
        self.stride = (stride, stride) if isinstance(stride, int) else tuple(stride)
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=self.patch_size,
            stride=self.stride,
            bias=bias,
        )
        self.norm = norm_layer(embed_dim) if norm_layer is not None else nn.Identity()
        self.grid_size = self._grid_size(self.img_size)
        self.num_patches = self.grid_size[0] * self.grid_size[1]

    def _grid_size(self, img_size):
        return (
            (img_size[0] - self.patch_size[0]) // self.stride[0] + 1,
            (img_size[1] - self.patch_size[1]) // self.stride[1] + 1,
        )

    def update_input_size(self, img_size):
        self.img_size = (img_size, img_size) if isinstance(img_size, int) else tuple(img_size)
        self.grid_size = self._grid_size(self.img_size)
        self.num_patches = self.grid_size[0] * self.grid_size[1]

    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x


def resize_pos_embed(pos_embed, target_grid):
    cls_token = pos_embed[:, :1, :]
    patch_pos = pos_embed[:, 1:, :]
    src_tokens = patch_pos.shape[1]
    src_grid = int(math.sqrt(src_tokens))
    if src_grid * src_grid != src_tokens:
        raise ValueError(f"Unexpected pos_embed token count: {src_tokens}")
    patch_pos = patch_pos.reshape(1, src_grid, src_grid, -1).permute(0, 3, 1, 2)
    patch_pos = F.interpolate(patch_pos.float(), size=target_grid, mode="bicubic", align_corners=False)
    patch_pos = patch_pos.permute(0, 2, 3, 1).reshape(1, target_grid[0] * target_grid[1], -1)
    return torch.cat([cls_token, patch_pos.to(pos_embed.dtype)], dim=1)


def resolve_vit_small_name():
    preferred = CONFIG["vit_model"]
    available_pretrained = set(timm.list_pretrained("vit_small_patch16_224*"))
    if preferred in available_pretrained:
        return preferred
    available_models = set(timm.list_models("vit_small_patch16_224*"))
    if preferred in available_models:
        return preferred
    fallback_candidates = [
        "vit_small_patch16_224.augreg_in21k_ft_in1k",
        "vit_small_patch16_224.augreg_in21k",
        "vit_small_patch16_224.dino",
        "vit_small_patch16_224",
    ]
    for candidate in fallback_candidates:
        if candidate in available_pretrained or candidate in available_models:
            print(f"Using fallback ViT-Small backbone: {candidate}")
            return candidate
    raise RuntimeError("No suitable timm ViT-Small/16 model found")


VIT_MODEL = resolve_vit_small_name()
print(f"Backbone: {VIT_MODEL}")


class TransReID(nn.Module):
    def __init__(
        self,
        num_classes,
        num_cameras=0,
        embed_dim=384,
        vit_model="vit_small_patch16_224.augreg_in21k_ft_in1k",
        pretrained=True,
        sie_camera=True,
        jpm=True,
        img_size=224,
        overlap_stride=12,
    ):
        super().__init__()
        self.sie_camera = sie_camera and num_cameras > 0
        self.jpm = jpm
        self.vit = timm.create_model(vit_model, pretrained=pretrained, num_classes=0, img_size=img_size)
        self.vit_dim = self.vit.embed_dim
        self.num_blocks = len(self.vit.blocks)

        old_patch_embed = self.vit.patch_embed
        overlap_patch_embed = OverlapPatchEmbed(
            img_size=img_size,
            patch_size=old_patch_embed.patch_size[0],
            stride=overlap_stride,
            in_chans=3,
            embed_dim=self.vit_dim,
            bias=old_patch_embed.proj.bias is not None,
            norm_layer=type(old_patch_embed.norm) if not isinstance(old_patch_embed.norm, nn.Identity) else None,
        )
        overlap_patch_embed.proj.weight.data.copy_(old_patch_embed.proj.weight.data)
        if old_patch_embed.proj.bias is not None and overlap_patch_embed.proj.bias is not None:
            overlap_patch_embed.proj.bias.data.copy_(old_patch_embed.proj.bias.data)
        self.vit.patch_embed = overlap_patch_embed
        self.vit.patch_embed.update_input_size((img_size, img_size))
        target_grid = self.vit.patch_embed.grid_size
        self.vit.pos_embed = nn.Parameter(resize_pos_embed(self.vit.pos_embed.data, target_grid))

        if self.sie_camera:
            self.sie_embed = nn.Parameter(torch.zeros(num_cameras, 1, self.vit_dim))
            nn.init.trunc_normal_(self.sie_embed, std=0.02)

        self.bn = nn.BatchNorm1d(self.vit_dim)
        self.bn.bias.requires_grad_(False)

        self.proj = nn.Linear(self.vit_dim, embed_dim, bias=False) if embed_dim != self.vit_dim else nn.Identity()
        self.cls_head = nn.Linear(embed_dim, num_classes, bias=False)
        if isinstance(self.proj, nn.Linear):
            nn.init.kaiming_normal_(self.proj.weight, mode="fan_out")
        nn.init.normal_(self.cls_head.weight, std=0.001)

        if self.jpm:
            self.bn_jpm = nn.BatchNorm1d(self.vit_dim)
            self.bn_jpm.bias.requires_grad_(False)
            self.jpm_cls = nn.Linear(self.vit_dim, num_classes, bias=False)
            nn.init.normal_(self.jpm_cls.weight, std=0.001)

        print(
            f"TransReID: {vit_model}, vit_dim={self.vit_dim}, embed_dim={embed_dim}, "
            f"SIE={self.sie_camera}({num_cameras}), JPM={self.jpm}, overlap_stride={overlap_stride}, "
            f"grid={self.vit.patch_embed.grid_size}"
        )

    def forward(self, x, cam_ids=None):
        batch_size = x.shape[0]
        x = self.vit.patch_embed(x)

        if hasattr(self.vit, "_pos_embed"):
            x = self.vit._pos_embed(x)
        else:
            cls_token = self.vit.cls_token.expand(batch_size, -1, -1)
            x = torch.cat([cls_token, x], dim=1) + self.vit.pos_embed
            if hasattr(self.vit, "pos_drop"):
                x = self.vit.pos_drop(x)

        if self.sie_camera and cam_ids is not None:
            x = x + self.sie_embed[cam_ids]

        if hasattr(self.vit, "patch_drop"):
            x = self.vit.patch_drop(x)
        if hasattr(self.vit, "norm_pre"):
            x = self.vit.norm_pre(x)

        for block in self.vit.blocks:
            x = block(x)
        x = self.vit.norm(x)

        global_feat = x[:, 0]
        bn_feat = self.bn(global_feat)
        proj_feat = self.proj(bn_feat)

        if self.training:
            cls_logits = self.cls_head(proj_feat)
            if self.jpm:
                patches = x[:, 1:]
                shuffle_index = torch.randperm(patches.size(1), device=x.device)
                shuffled = patches[:, shuffle_index]
                midpoint = shuffled.size(1) // 2
                jpm_feat = (shuffled[:, :midpoint].mean(1) + shuffled[:, midpoint:].mean(1)) / 2
                return cls_logits, global_feat, self.jpm_cls(self.bn_jpm(jpm_feat))
            return cls_logits, global_feat

        return F.normalize(proj_feat, p=2, dim=1)

    def get_llrd_param_groups(self, backbone_lr, head_lr, decay=0.75):
        groups = {}
        for name, param in self.named_parameters():
            if not param.requires_grad:
                continue
            if name.startswith("vit."):
                if "blocks." in name:
                    block_index = int(name.split("blocks.")[1].split(".")[0])
                    depth = block_index + 1
                elif any(key in name for key in ["patch_embed", "cls_token", "pos_embed", "norm_pre"]):
                    depth = 0
                else:
                    depth = self.num_blocks + 1
                scale = decay ** (self.num_blocks + 1 - depth)
                lr = backbone_lr * scale
                group_key = f"bb_{depth}"
            else:
                lr = head_lr
                group_key = "head"
            if group_key not in groups:
                groups[group_key] = {"params": [], "lr": lr}
            groups[group_key]["params"].append(param)
        return sorted(groups.values(), key=lambda item: item["lr"])


model = TransReID(
    num_classes=num_classes,
    num_cameras=num_cameras,
    embed_dim=CONFIG["embed_dim"],
    vit_model=VIT_MODEL,
    pretrained=True,
    sie_camera=True,
    jpm=True,
    img_size=CONFIG["img_size"],
    overlap_stride=CONFIG["overlap_patch_stride"],
).to(DEVICE)

if NUM_GPUS > 1:
    model = nn.DataParallel(model)
    print(f"Wrapped in DataParallel ({NUM_GPUS} GPUs)")

print(f"Parameters: {sum(parameter.numel() for parameter in model.parameters()):,}")

## 4. Training

This follows the requested ViT-Small recipe directly: CE + Triplet + delayed Center loss, AdamW, cosine annealing, and CityFlowV2 evaluation with mAP / Rank-1 / Rank-5 / Rank-10.

In [ ]:
ce_loss = CrossEntropyLabelSmooth(num_classes, CONFIG["label_smoothing"]).to(DEVICE)
triplet_loss = TripletLossHardMining(margin=CONFIG["triplet_margin"]).to(DEVICE)
center_loss = CenterLoss(num_classes, model.module.vit_dim if hasattr(model, "module") else model.vit_dim).to(DEVICE)

raw_model = model.module if hasattr(model, "module") else model
param_groups = raw_model.get_llrd_param_groups(
    CONFIG["backbone_lr"],
    CONFIG["head_lr"],
    decay=CONFIG["llrd_decay"],
)
optimizer = torch.optim.AdamW(param_groups, weight_decay=CONFIG["weight_decay"])
center_optimizer = torch.optim.SGD(center_loss.parameters(), lr=0.5)
base_lrs = [group["lr"] for group in optimizer.param_groups]
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=110)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")

history = {
    "loss": [],
    "mAP": [],
    "rank1": [],
    "rank5": [],
    "rank10": [],
    "epochs": [],
}
best_metrics = {"mAP": 0.0, "rank1": 0.0, "rank5": 0.0, "rank10": 0.0, "epoch": -1}

print("=" * 70)
print(f"  Training ViT-Small TransReID on CityFlowV2 ({num_classes} IDs, {num_cameras} cameras)")
print(f"  Backbone: {VIT_MODEL}")
print(
    f"  Losses: CE(eps={CONFIG['label_smoothing']}) + Triplet(m={CONFIG['triplet_margin']}) "
    f"+ Center(w={CONFIG['center_weight']}, start={CONFIG['center_start_epoch']})"
)
print(
    f"  LR: backbone={CONFIG['backbone_lr']:.2e}, head={CONFIG['head_lr']:.2e}, "
    f"LLRD={CONFIG['llrd_decay']}"
)
print(
    f"  Batch: P={CONFIG['p']}, K={CONFIG['k']} ({CONFIG['batch_size']} total) | "
    f"Epochs={CONFIG['epochs']} | Warmup={CONFIG['warmup_epochs']}"
)
print("=" * 70)

train_start = time.time()
for epoch in range(CONFIG["epochs"]):
    model.train()
    use_center = epoch >= CONFIG["center_start_epoch"]
    running_loss = 0.0
    num_batches = 0

    if epoch < CONFIG["warmup_epochs"]:
        warmup_factor = (epoch + 1) / CONFIG["warmup_epochs"]
        for group_index, group in enumerate(optimizer.param_groups):
            group["lr"] = base_lrs[group_index] * warmup_factor
    elif epoch == CONFIG["warmup_epochs"]:
        for group_index, group in enumerate(optimizer.param_groups):
            group["lr"] = base_lrs[group_index]

    for imgs, pids, cams, _ in train_loader:
        imgs = imgs.to(DEVICE)
        pids = pids.to(DEVICE).long()
        cams = cams.to(DEVICE).long()

        optimizer.zero_grad()
        if use_center:
            center_optimizer.zero_grad()

        with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
            outputs = model(imgs, cam_ids=cams)
            if len(outputs) == 3:
                cls_logits, global_feat, jpm_logits = outputs
                total_loss = ce_loss(cls_logits, pids) + triplet_loss(global_feat, pids)
                total_loss = total_loss + CONFIG["jpm_loss_weight"] * ce_loss(jpm_logits, pids)
            else:
                cls_logits, global_feat = outputs
                total_loss = ce_loss(cls_logits, pids) + triplet_loss(global_feat, pids)

            if use_center:
                center_term = center_loss(global_feat.float(), pids)
                total_loss = total_loss + CONFIG["center_weight"] * center_term

        scaler.scale(total_loss).backward()
        scaler.unscale_(optimizer)
        if use_center:
            scaler.unscale_(center_optimizer)
        torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=CONFIG["grad_clip_norm"])
        scaler.step(optimizer)
        if use_center:
            scaler.step(center_optimizer)
        scaler.update()

        if not torch.isnan(total_loss):
            running_loss += float(total_loss.detach().item())
        num_batches += 1

    if epoch >= CONFIG["warmup_epochs"]:
        scheduler.step()

    epoch_loss = running_loss / max(num_batches, 1)
    history["loss"].append(epoch_loss)

    if (epoch + 1) % 10 == 0:
        current_head_lr = optimizer.param_groups[-1]["lr"]
        center_tag = " [+center]" if use_center else ""
        print(f"Epoch {epoch + 1:3d} | Loss {epoch_loss:.4f} | head_lr={current_head_lr:.2e}{center_tag}")

    if (epoch + 1) % CONFIG["eval_every"] == 0 or epoch == CONFIG["epochs"] - 1:
        metrics = evaluate_cityflow(model, query_loader, gallery_loader, device=DEVICE)
        history["epochs"].append(epoch + 1)
        history["mAP"].append(metrics["mAP"])
        history["rank1"].append(metrics["rank1"])
        history["rank5"].append(metrics["rank5"])
        history["rank10"].append(metrics["rank10"])

        is_best = metrics["mAP"] > best_metrics["mAP"]
        if is_best:
            best_metrics = {**metrics, "epoch": epoch + 1}
            payload = {
                "model_state_dict": raw_model.state_dict(),
                "epoch": epoch + 1,
                "mAP": metrics["mAP"],
                "rank1": metrics["rank1"],
                "config": {
                    **CONFIG,
                    "num_classes": num_classes,
                    "num_cameras": num_cameras,
                    "resolved_vit_model": VIT_MODEL,
                },
            }
            torch.save(payload, CHECKPOINT_PATH)

        print(
            f"Eval @ epoch {epoch + 1:3d}: mAP={metrics['mAP']:.4f}, R1={metrics['rank1']:.4f}, "
            f"R5={metrics['rank5']:.4f}, R10={metrics['rank10']:.4f}{' BEST' if is_best else ''}"
        )

last_payload = {
    "model_state_dict": raw_model.state_dict(),
    "epoch": CONFIG["epochs"],
    "mAP": best_metrics["mAP"],
    "rank1": best_metrics["rank1"],
    "config": {
        **CONFIG,
        "num_classes": num_classes,
        "num_cameras": num_cameras,
        "resolved_vit_model": VIT_MODEL,
    },
}
torch.save(last_payload, LAST_CHECKPOINT_PATH)

metadata = {
    "task": "vehicle_reid",
    "dataset": "cityflowv2",
    "training": "TransReID ViT-Small/16 IN21k -> CityFlowV2 fine-tune",
    "model": {
        "architecture": VIT_MODEL,
        "type": "transreid",
        "tricks": ["SIE", "JPM", "OverlapPatchEmbed", "BNNeck", "CE+LS(0.1)", "Triplet(m=0.3)", "CenterLoss", "CosineAnnealingLR", "AdamW"],
        "embedding_dim": CONFIG["embed_dim"],
        "input_size": [CONFIG["img_size"], CONFIG["img_size"]],
        "normalization": {"mean": CONFIG["imagenet_mean"], "std": CONFIG["imagenet_std"]},
        "overlap_patch_stride": CONFIG["overlap_patch_stride"],
        "num_cameras": num_cameras,
        "num_classes": num_classes,
    },
    "best": best_metrics,
    "split": {
        "train_images": len(train_data),
        "train_ids": num_classes,
        "query_images": len(query_data),
        "gallery_images": len(gallery_data),
        "eval_ids": len(eval_ids),
    },
    "files": {
        "best_checkpoint": str(CHECKPOINT_PATH),
        "last_checkpoint": str(LAST_CHECKPOINT_PATH),
    },
    "config": {**CONFIG, "resolved_vit_model": VIT_MODEL},
}
with open(METADATA_EXPORT_PATH, "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)

elapsed_hours = (time.time() - train_start) / 3600.0
print(f"Training finished in {elapsed_hours:.2f}h")
print(f"Best checkpoint: {CHECKPOINT_PATH}")
print(json.dumps(best_metrics, indent=2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("09k ViT-Small TransReID -- CityFlowV2", fontsize=14, fontweight="bold")

axes[0].plot(history["loss"], color="tab:blue", alpha=0.8)
axes[0].set_title("Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.3)

if history["epochs"]:
    axes[1].plot(history["epochs"], [value * 100 for value in history["mAP"]], "o-", label="mAP")
    axes[1].plot(history["epochs"], [value * 100 for value in history["rank1"]], "s-", label="Rank-1")
    axes[1].plot(history["epochs"], [value * 100 for value in history["rank5"]], "^-", label="Rank-5")
    axes[1].plot(history["epochs"], [value * 100 for value in history["rank10"]], "d-", label="Rank-10")
axes[1].set_title("CityFlowV2 Metrics")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Percent")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(CURVE_PATH, dpi=150, bbox_inches="tight")
plt.show()

print(f"Curves saved to {CURVE_PATH}")
print(f"Best checkpoint exists: {CHECKPOINT_PATH.exists()}")
print(f"Metadata exists: {METADATA_EXPORT_PATH.exists()}")